# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

# 🌐 1. Environment Setup


In [ ]:
# ── Cell 1: 💢 Environment Sync & Cache Exorcism ────────────
!pip install -q -U bitsandbytes triton transformers
import os, shutil, sys

# 1. Nuclear Sync: Ensure fresh library code
REPO_NAME = 'EFV-nn'
if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
!git clone https://github.com/ey3lock3r/EFV-nn.git
if f'/kaggle/working/{REPO_NAME}/src' not in sys.path: sys.path.insert(0, f'/kaggle/working/{REPO_NAME}/src')

# 2. Path Configuration
SAVE_PATH = '/kaggle/working/ppc_3b_pretrain.pt'

# 3. Corruption Purge & Cache Wipe
if os.path.exists(SAVE_PATH) and os.path.getsize(SAVE_PATH) < 5 * 1024**3:
    print(f'🗑️ Purging corrupted remnant ({os.path.getsize(SAVE_PATH)/1e9:.2f} GB)')
    os.remove(SAVE_PATH)
if os.path.exists(SAVE_PATH + '.tmp'): os.remove(SAVE_PATH + '.tmp')

for p in ['/tmp/torchinductor_root', os.path.expanduser('~/.cache/torch')]:
    if os.path.exists(p): shutil.rmtree(p)

# 4. Checkpoint Discovery
candidates = [f for f in os.listdir('.') if f.endswith('.pt') and f != 'ppc_3b_pretrain.pt']
healthy = [f for f in candidates if os.path.getsize(f) > 5 * 1024**3]
LOAD_PATH = max(healthy, key=os.path.getsize) if healthy else SAVE_PATH

print(f'✅ Environment Ready. Load Target: {LOAD_PATH}')


# 🔑 2. Secrets & Monitoring


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print("✅ Logged into W&B and HF")
except Exception as e:
    print(f"⚠️ Secrets failure: {e}. Training will continue without W&B.")


# 📦 3. Data Pipeline


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ────────────
from datasets import load_dataset
import torch

def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex['text'])['input_ids'])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


# 📡 4. W&B Rescue (Optional)


In [ ]:
# ── Cell 4: Optional 🛟 W&B Rescue (Pull checkpoint from cloud) ──────────
import shutil, os, wandb
RESCUE_RUN_PATH = "dhinsresearch/ppc-3b-dynamic/YOUR_RUN_ID" # Update this
RESCUE_FILE = "ppc_3b_pretrain.pt"

def rescue_checkpoint(run_path, filename, target_path):
    print(f"📡 Synching {filename} from {run_path}...")
    try:
        restored_file = wandb.restore(filename, run_path=run_path)
        source_path = os.path.abspath(restored_file.name)
        target_path = os.path.abspath(target_path)
        if source_path != target_path:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            shutil.move(source_path, target_path)
            print(f"🚚 Moved from {source_path} → {target_path}")
        print(f"✅ Rescue Successful! Ready at: {target_path}")
    except Exception as e:
        print(f"❌ Rescue Failed: {e}")

# rescue_checkpoint(RESCUE_RUN_PATH, RESCUE_FILE, SAVE_PATH)


# 🏗️ 5. Architectural Setup


In [ ]:
# ── Cell 5: 🏗️ Architectural Setup & State Loading ────────────
import torch, gc, importlib, os
import efv_nn.triton_kernels, efv_nn.ppc_sharded, efv_nn.ppc_gnn, efv_nn.ppc_core
from efv_nn.ppc_sharded import ShardedPPCGraphLLM
from transformers import AutoTokenizer

# 1. Hot-Reload Library
for m in [efv_nn.triton_kernels, efv_nn.ppc_core, efv_nn.ppc_gnn, efv_nn.ppc_sharded]: importlib.reload(m)

# 2. Configuration
VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
PRIME_DELAYS = [1, 2, 3, 5, 7, 11, 13, 17]
RESUME = True

# 3. Model Instantiation
print('🏗️ Instantiating 3.75B Sharded PPC-GNN...')
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', token=os.environ.get('HF_TOKEN'))
gc.collect(); torch.cuda.empty_cache()
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, use_jacobian=True, prime_delays=PRIME_DELAYS)

# 4. Safety Guards (The Ghost Inspector)
moe_type = str(type(model.layers[0].moe))
is_compiled = any(hasattr(l, '_orig_mod') for l in model.layers) or hasattr(model, '_orig_mod') or 'Optimized' in moe_type
if is_compiled:
    print(f'❌ GHOST DETECTED: Model/MoE is {moe_type}')
    raise RuntimeError('⚠️ RESTART KERNEL: Stale compiled references detected.')
print(f'✅ Memory Sanity: Native State Verified.')

# 5. Flexible State-Dict Loading (Ghost-Blind Mapping)
if RESUME and os.path.exists(LOAD_PATH):
    print(f'🔄 Loading: {LOAD_PATH} ({os.path.getsize(LOAD_PATH)/1e9:.2f} GB)')
    ckpt = torch.load(LOAD_PATH, map_location='cpu')
    
    clean_dict = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    target_dict = model.state_dict()
    final_dict = {kt: clean_dict[kt.replace('_orig_mod.', '')] for kt in target_dict if kt.replace('_orig_mod.', '') in clean_dict}
    
    msg = model.load_state_dict(final_dict, strict=False)
    START_STEP = ckpt.get('step', 0)
    print(f'✅ Resumed Step {START_STEP}. Missing: {len(msg.missing_keys)} (APD/OCNS defaults used).')
else:
    START_STEP = 0
    print('✨ Starting Step 0 (No checkpoint found).')

triton_ok = getattr(model.layers[0], '_triton_available', False)
print(f'⚡ Triton: {"ACTIVE (Hyper-Drive)" if triton_ok else "FALLBACK"}')
print(f'✅ Model Ready. Total Params: {model.total_params/1e9:.2f}B')


# 🔬 6. Model Identity Audit


In [ ]:
# ── Cell 6: 🔬 Model Identity Audit (Sanity Check) ──
def run_identity_audit(model, path):
    def _audit_weights(label, weights):
        w_mean = weights.abs().mean().item()
        w_std  = weights.std().item()
        status = "✅ LEARNED" if w_std > 0.05 else "❌ RANDOM/NOISE"
        print(f"[{label}] Mean: {w_mean:.6f} | Std: {w_std:.6f} | Status: {status}")
        return w_std > 0.05

    print("🔍 Auditing Memory vs Disk...")
    # 1. Memory
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    memory_learned = _audit_weights("MEMORY", raw_model.embedding.weight)
    
    # 2. Disk
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location='cpu')
            s_dict = ckpt['model_state'] if 'model_state' in ckpt else ckpt
            w_disk = s_dict.get('embedding.weight') or s_dict.get('_orig_mod.embedding.weight')
            if w_disk is not None:
                _audit_weights("DISK  ", w_disk.float())
            else:
                print("⚠️ DISK: Could not find 'embedding.weight' in checkpoint.")
        except Exception as e:
            print(f"⚠️ DISK: Failed to read checkpoint: {e}")
    else:
        print("⚪ DISK: No checkpoint found at target path.")
    
    if not memory_learned:
        print("\n❗ WARNING: Model in memory is RANDOM. Please verify your load logic.")

# 🚀 To perform a sanity check, uncomment the line below:
# run_identity_audit(model, LOAD_PATH)


# 🚀 7. Production Training Loop


In [ ]:
# ── Cell 7: 🚀 Start Training Loop (Stabilized) ────────────
import time, wandb, shutil, gc, math, torch
import bitsandbytes.optim as bnb
import torch.nn.functional as F
from efv_nn.ppc_gnn import spectral_guardian_penalty

ENABLE_SPECTRAL_GUARDIAN = True
DEVICE1 = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'

def save_checkpoint(model, step, path, avg_energy=0.0, energy_threshold=1.0, sync_wandb=True):
    if avg_energy > energy_threshold:
        print(f'🛑 ABORTED: Energy {avg_energy:.4f} too high. Protecting healthy checkpoint.')
        return
    
    gc.collect(); torch.cuda.empty_cache()
    if os.path.exists(path):
        # Move instead of copy to save 7.5GB of disk space on Kaggle
        shutil.move(path, path.replace('.pt', '_prev.pt'))

    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    state_dict = raw_model.state_dict()
    compressed = {k: v.float().half() if v.dtype == torch.float32 else v for k, v in state_dict.items()}
    
    torch.save({'step': step, 'model_state': compressed}, path + '.tmp')
    os.replace(path + '.tmp', path)
    if sync_wandb: wandb.save(path, base_path=os.path.dirname(path))
    print(f'✅ Step {step} saved.')

def train(model, tokenizer, num_steps=1000, start_step=0, local_iters=24, lr=8e-5):
    run = wandb.init(project='ppc-3b-dynamic', name=f'ppc-3b-{time.strftime("%H%M")}')
    dataloader = get_dataloader(tokenizer, batch_size=2, seq_len=256)
    
    # Split params for OCNS causal memory vs base weights
    delay_params = [p for n, p in model.named_parameters() if 'delay_gains' in n]
    base_params = [p for n, p in model.named_parameters() if 'delay_gains' not in n]
    
    opt = bnb.PagedAdamW8bit([
        {'params': base_params, 'lr': lr},
        {'params': delay_params, 'lr': 5e-4}
    ])

    t0 = time.time(); last_step = start_step
    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)

        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, avg_iters, avg_energy, layer_energies = model(ids, local_iters=local_iters)
            loss = F.cross_entropy(logits.reshape(-1, 128256), targets.reshape(-1))
            if ENABLE_SPECTRAL_GUARDIAN: loss += spectral_guardian_penalty(layer_energies.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % 5 == 0:
            dt = (time.time() - t0) * 1000 / (step - last_step) if step > last_step else 0
            t0 = time.time(); last_step = step
            wandb.log({'loss': loss.item(), 'avg_iters': avg_iters, 'duration_ms': dt}, step=step)
            print(f'St {step} | L: {loss.item():.4f} | E: {avg_energy.item():.4f} | D: {dt:.0f}ms')

        if (step + 1) % 100 == 0: save_checkpoint(model, step+1, '/kaggle/working/ppc_3b_pretrain.pt', avg_energy.item())
        if step >= (start_step + num_steps): break
    wandb.finish()


# 📈 8. Training Phases


In [ ]:
# ── Cell 8: 🚀 Start Training Loop (Axiomatic Phases) ──

# Phase 1: 🛡️ Stabilization (16 iters, 1.5e-4, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=500, local_iters=16, lr=1.5e-4)

# Phase 2: 🚀 Depth Scaling (24 iters, 1.5e-4, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=2000, local_iters=24, lr=1.5e-4)

# Phase 3: 💎 Precision Polish (32 iters, 5e-5, 256x Throttled)
# train(model, tokenizer, start_step=START_STEP, num_steps=5000, local_iters=32, lr=5e-5)

# Phase 4: 🧠 Semantic Emergence (32 iters, 5e-5, RELEASED GRADIENTS)
# Completed at Step 10,000. E climbed to 0.0034 — plateau confirmed.
# train(model, tokenizer, start_step=START_STEP, num_steps=10000, local_iters=32, lr=5e-5)

# Phase 5: 🔬 Deep Resonance Tuning (48 iters, 1e-5, RELEASED GRADIENTS)
# Breaking the phasal plateau. Expected L: 8.0 → 7.6
# train(model, tokenizer, start_step=START_STEP, num_steps=10000, local_iters=48, lr=1e-5)

# Phase 6: 💠 Phasal Annealing (48 iters, 1e-6, RELEASED GRADIENTS)
# Final precision collapse. Targets: L < 7.3, E < 0.002. 
train(model, tokenizer, start_step=START_STEP, num_steps=10000, local_iters=48, lr=1e-6)


# 🧠 9. Instant Verification


In [ ]:
# ── Cell 9: 🗣️ Verification (Instant) ──
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    out_reg = model.generate(inputs, max_new_tokens=40, local_iters=32)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    out_swarm = model.generate_swarm(inputs, max_new_tokens=40, swarm_size=4, local_iters=32)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")
    print("\n🧬 Running MoE Health Audit...")
    captured_indices = []
    def hook_fn(module, input, output):
        captured_indices.append(output[1].detach().cpu())
        
    hooks = []
    try:
        for name, module in model.named_modules():
            if module.__class__.__name__ == 'ExpertChoiceMoEMatcher':
                hooks.append(module.register_forward_hook(hook_fn))
                
        with torch.no_grad(), torch.amp.autocast('cuda'):
            model(inputs, local_iters=8)
            

    finally:
        for h in hooks: h.remove()
    
    B_T = inputs.shape[1]
    total_coverage = 0
    for indices in captured_indices[:24]:
        flat_indices = indices.flatten()
        unique_tokens = torch.unique(flat_indices).shape[0]
        total_coverage += (unique_tokens / float(B_T))
            
    avg_cov = total_coverage / len(captured_indices[:24])
    print(f"🔬 Average Expert Diversity Score (Coverage): {avg_cov:.1%}")
    if avg_cov < 0.3:
        print("🚨 WARNING: High Expert Cloning detected. Experts are picking the same tokens.")
    elif avg_cov > 0.8:
        print("✅ HEALTHY: Maximum MoE utilization. Experts are highly specialized.")
    else:
        print("⚠️ MODERATE: Experts are learning, but there is noticeable overlap.")


# To run: 
# verify_generation(model, tokenizer)


# 🗣️ 10. Generation Sandbox


In [ ]:
# ── Cell 10: 🧪 Generation Sandbox (Verification) ────────────
def run_sandbox_verification(prompt='The fundamental principles of biology include'):
    inputs = tokenizer(prompt, return_tensors='pt')['input_ids']
    print('\n🧠 Running Native Generation...')
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=50, local_iters=32)
    print(f'Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}')

# run_sandbox_verification()
